# FonoKids -- Entrenar el modelo de video con TODO el rostro

Entrena la **misma arquitectura TCN + Conformer** de `scripts/train_landmarks_transformer.py`, pero con los landmarks de la **cara completa** (68 puntos, 408 numeros por frame) que saca `Extraer_Cara_Completa_FonoKids.ipynb`.

Para que la comparacion con el modelo de solo labios sea justa se mantiene todo igual: **mismos hiperparametros, misma semilla y mismo reparto 80/20**, sobre las clases y clips que hayas extraido. Lo unico que cambia es la entrada (de 120 a 408 numeros por frame; el tamano de entrada se detecta solo desde los `.npy`).

**Una sola ruta**: pega en `RUTA_VIDEOS` la MISMA carpeta que usaste en la extraccion; los landmarks, el modelo y los resultados se buscan y guardan junto a ella (`modelos_cara_completa/`, `resultados_cara_completa/`). No pisa nada: tu `best.pth` de solo labios queda intacto.

Al final muestra Top-1, Top-3 y acierto bien/mal, junto a las cifras del modelo de solo labios para comparar. Funciona en Colab y en tu PC.

## 1. Entorno

In [ ]:
import os, re, sys, json, math, glob, subprocess

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
print('Corriendo en:', 'Google Colab' if EN_COLAB else 'tu PC (local)')

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


def instalar_si_falta(modulo, paquete):
    try:
        __import__(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


import torch
print('CUDA disponible:', torch.cuda.is_available(), '(en Colab: Entorno de ejecucion > Cambiar tipo > GPU)')

import numpy as np
import unicodedata
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

## 2. Rutas

In [ ]:
RUTA_VIDEOS = '/content/drive/MyDrive/Aplicaciones_Inteligentes/SACA LANDMARKS/datos/sesiones_continuas'


def buscar_raiz_local():
    actual = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(actual, 'data')) and os.path.isdir(os.path.join(actual, 'scripts')):
            return actual
        actual = os.path.dirname(actual)
    return r'D:\FonoKids'


if not os.path.isdir(RUTA_VIDEOS):
    RUTA_VIDEOS = os.path.join(buscar_raiz_local(), 'data', 'sesiones_continuas')
RUTA_VIDEOS = RUTA_VIDEOS.rstrip('/\\')
CARPETA_SALIDA = os.path.dirname(RUTA_VIDEOS)
assert os.path.isdir(RUTA_VIDEOS), f'No encontre {RUTA_VIDEOS}: cambia RUTA_VIDEOS por la carpeta de tu Drive que tiene los videos'
print('Videos en:     ', RUTA_VIDEOS)
print('Resultados en: ', CARPETA_SALIDA)

LANDMARKS_DIR = os.path.join(CARPETA_SALIDA, 'landmarks_cara_completa_npy')
MODELS_DIR = os.path.join(CARPETA_SALIDA, 'modelos_cara_completa')
OUTPUTS_DIR = os.path.join(CARPETA_SALIDA, 'resultados_cara_completa')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

assert os.path.isdir(LANDMARKS_DIR), f'No encontre {LANDMARKS_DIR}: corre primero Extraer_Cara_Completa_FonoKids.ipynb'
print('Landmarks de cara completa:', LANDMARKS_DIR)
_conteo = {c: len(glob.glob(os.path.join(LANDMARKS_DIR, c, '*.npy'))) for c in sorted(os.listdir(LANDMARKS_DIR)) if os.path.isdir(os.path.join(LANDMARKS_DIR, c))}
print(f'{len(_conteo)} clases, {sum(_conteo.values())} clips.')
_pocas = {c: n for c, n in _conteo.items() if n < 20}
if _pocas:
    print('OJO: clases con menos de 20 clips (el modelo las aprende mal):', _pocas)
print('El modelo se guardara en:', MODELS_DIR)

## 3. Dataset (identico a `lip_reading_dataset.py`)

In [ ]:
def _nfc(text):
    return unicodedata.normalize('NFC', text)


class LipReadingDataset(Dataset):
    def __init__(self, data_dir, max_frames=60, file_paths=None, augment=False):
        self.data_dir = data_dir
        self.max_frames = max_frames
        self.augment = augment

        label_map_path = os.path.join(data_dir, 'label_map.json')
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f'No encontre {label_map_path}. Corre el notebook de extraccion primero.')
        with open(label_map_path, 'r', encoding='utf-8') as f:
            raw_label_map = json.load(f)
        self.label_map = {_nfc(word): idx for word, idx in raw_label_map.items()}

        num_classes = max(self.label_map.values()) + 1
        self.class_names = ['?'] * num_classes
        for word, idx in self.label_map.items():
            self.class_names[idx] = word

        if file_paths is not None:
            self.file_paths = list(file_paths)
        else:
            self.file_paths = sorted(glob.glob(os.path.join(data_dir, '*', '*.npy')))
        if not self.file_paths:
            raise RuntimeError(f'No hay archivos .npy en {data_dir}.')

        first_sample = np.load(self.file_paths[0])
        self.landmark_dim = int(first_sample.shape[-1])

    def __len__(self):
        return len(self.file_paths)

    def _label_from_path(self, path):
        word = _nfc(os.path.basename(os.path.dirname(path)))
        return self.label_map[word]

    def _apply_augmentation(self, sequence):
        sequence = sequence + np.random.normal(0, 0.003, sequence.shape).astype(np.float32)
        shift = np.random.uniform(-0.01, 0.01, size=(1, sequence.shape[1])).astype(np.float32)
        scale = np.random.uniform(0.97, 1.03)
        sequence = sequence * scale + shift
        T = sequence.shape[0]
        n_mask = max(1, int(round(T * np.random.uniform(0.08, 0.12))))
        n_mask = min(n_mask, T)
        mask_idx = np.random.choice(T, size=n_mask, replace=False)
        sequence[mask_idx] = 0.0
        return sequence.astype(np.float32)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        sequence = np.load(path).astype(np.float32)
        label = self._label_from_path(path)
        if self.augment:
            sequence = self._apply_augmentation(sequence)
        T = sequence.shape[0]
        if T < self.max_frames:
            pad = np.zeros((self.max_frames - T, sequence.shape[1]), dtype=np.float32)
            sequence = np.concatenate([sequence, pad], axis=0)
            real_length = T
        elif T > self.max_frames:
            sequence = sequence[: self.max_frames]
            real_length = self.max_frames
        else:
            real_length = T
        return (
            torch.from_numpy(sequence),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(real_length, dtype=torch.long),
        )


def build_dataloaders(data_dir, max_frames=60, batch_size=32, val_ratio=0.2, seed=42, augment_train=True):
    reference = LipReadingDataset(data_dir=data_dir, max_frames=max_frames)
    file_paths = reference.file_paths

    labels_by_path = {p: reference.label_map[_nfc(os.path.basename(os.path.dirname(p)))] for p in file_paths}
    paths_by_class = {}
    for p in file_paths:
        paths_by_class.setdefault(labels_by_path[p], []).append(p)

    rng = np.random.default_rng(seed)
    train_paths, val_paths = [], []
    for label, paths in paths_by_class.items():
        paths = list(paths)
        rng.shuffle(paths)
        n_val = max(1, int(round(len(paths) * val_ratio)))
        val_paths.extend(paths[:n_val])
        train_paths.extend(paths[n_val:])
    rng.shuffle(train_paths)
    rng.shuffle(val_paths)

    train_ds = LipReadingDataset(data_dir=data_dir, max_frames=max_frames, file_paths=train_paths, augment=augment_train)
    val_ds = LipReadingDataset(data_dir=data_dir, max_frames=max_frames, file_paths=val_paths, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

    return reference, train_loader, val_loader

## 4. Modelo TCN + Conformer (identico a `train_landmarks_transformer.py`)

In [ ]:
import time


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        residual = x
        x = x.transpose(1, 2)
        if key_padding_mask is not None:
            x = x.masked_fill(key_padding_mask.unsqueeze(1), 0.0)
        x = self.dropout(self.relu(self.bn1(self.conv1(x))))
        x = self.bn2(self.conv2(x))
        x = x.transpose(1, 2)
        return self.relu(residual + self.dropout(x))


class TCNFeatureExtractor(nn.Module):
    def __init__(self, channels, n_blocks=3, kernel_size=3, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TCNBlock(channels, kernel_size=kernel_size, dilation=2 ** i, dropout=dropout)
            for i in range(n_blocks)
        ])

    def forward(self, x, key_padding_mask=None):
        for block in self.blocks:
            x = block(x, key_padding_mask=key_padding_mask)
        return x


class ConformerFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=15, dropout=0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.pointwise_conv1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.glu = nn.GLU(dim=1)
        padding = (kernel_size - 1) // 2
        self.depthwise_conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, padding=padding, groups=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.swish = nn.SiLU()
        self.pointwise_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        residual = x
        x = self.layer_norm(x)
        x = x.transpose(1, 2)
        if key_padding_mask is not None:
            x = x.masked_fill(key_padding_mask.unsqueeze(1), 0.0)
        x = self.pointwise_conv1(x)
        x = self.glu(x)
        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = self.swish(x)
        x = self.pointwise_conv2(x)
        x = self.dropout(x)
        x = x.transpose(1, 2)
        return residual + x


class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dim_feedforward, conv_kernel_size=15, dropout=0.1):
        super().__init__()
        self.ff1 = ConformerFeedForward(d_model, dim_feedforward, dropout)
        self.self_attn_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)
        self.conv_module = ConformerConvModule(d_model, conv_kernel_size, dropout)
        self.ff2 = ConformerFeedForward(d_model, dim_feedforward, dropout)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, src_key_padding_mask=None):
        x = x + 0.5 * self.ff1(x)
        residual = x
        x_norm = self.self_attn_norm(x)
        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm, key_padding_mask=src_key_padding_mask, need_weights=False)
        x = residual + self.attn_dropout(attn_out)
        x = self.conv_module(x, key_padding_mask=src_key_padding_mask)
        x = x + 0.5 * self.ff2(x)
        return self.final_norm(x)


class ConformerEncoder(nn.Module):
    def __init__(self, d_model, n_layers, n_heads, dim_feedforward, conv_kernel_size=15, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, dim_feedforward, conv_kernel_size, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, src_key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        return x


class LipReadingConformer(nn.Module):
    def __init__(self, num_classes, input_dim, hidden_dim=128, n_layers=2, n_heads=8,
                 dim_feedforward=256, conv_kernel_size=15, tcn_blocks=3, tcn_kernel_size=3, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.tcn = TCNFeatureExtractor(hidden_dim, n_blocks=tcn_blocks, kernel_size=tcn_kernel_size, dropout=dropout)
        self.pos_encoder = PositionalEncoding(hidden_dim)
        self.conformer_encoder = ConformerEncoder(
            d_model=hidden_dim, n_layers=n_layers, n_heads=n_heads,
            dim_feedforward=dim_feedforward, conv_kernel_size=conv_kernel_size, dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward_features(self, x, src_key_padding_mask=None):
        x = self.input_proj(x)
        x = self.tcn(x, key_padding_mask=src_key_padding_mask)
        x = self.pos_encoder(x)
        x = self.conformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        return masked_mean_pool(x, src_key_padding_mask)

    def forward(self, x, src_key_padding_mask=None):
        pooled = self.forward_features(x, src_key_padding_mask=src_key_padding_mask)
        return self.classifier(pooled)


def masked_mean_pool(x, src_key_padding_mask):
    if src_key_padding_mask is None:
        return x.mean(dim=1)
    real_mask = (~src_key_padding_mask).unsqueeze(-1).float()
    summed = (x * real_mask).sum(dim=1)
    counts = real_mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


def make_padding_mask(lengths, max_len, device):
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for sequences, labels, lengths in loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
            mask = make_padding_mask(lengths, sequences.shape[1], device)
            if train:
                optimizer.zero_grad()
            outputs = model(sequences, src_key_padding_mask=mask)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * sequences.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    avg_loss = total_loss / total if total > 0 else 0.0
    acc = 100.0 * correct / total if total > 0 else 0.0
    return avg_loss, acc


def build_warmup_cosine_scheduler(optimizer, epochs, warmup_epochs=5):
    warmup_epochs = max(1, min(warmup_epochs, epochs - 1)) if epochs > 1 else 1

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 5. Entrenar

Mismos hiperparametros que el modelo de solo labios (50 epocas con early stopping, lote 32, 60 frames, lr 1e-3, label smoothing 0.1).

In [ ]:
VIDEO_MODEL_PATH = os.path.join(MODELS_DIR, 'best_cara_completa.pth')
SEED = 42
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Entrenando en: {device}')

EPOCHS = 50
BATCH_SIZE = 32
MAX_FRAMES = 60
LR = 1e-3
WEIGHT_DECAY = 1e-3
VAL_RATIO = 0.2
# Con 54 clases la pérdida de validación salta mucho mientras la tasa de aprendizaje es alta; con paciencia 15 el entrenamiento se cortaba en la época 27,
# antes de que el decaimiento coseno lo estabilizara. Se usa la paciencia = nº de épocas (sin parada anticipada) y se conserva el mejor modelo por pérdida de validación.
EARLY_STOPPING_PATIENCE = EPOCHS
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.1

dataset, train_loader, val_loader = build_dataloaders(
    data_dir=LANDMARKS_DIR, max_frames=MAX_FRAMES, batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO, seed=SEED, augment_train=True,
)
num_classes = len(dataset.class_names)
input_dim = dataset.landmark_dim
print(f'Clases ({num_classes}): {dataset.class_names}')
print(f'Dimension de entrada detectada: {input_dim}')
print(f'Total muestras: {len(dataset)} | Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}')

model = LipReadingConformer(num_classes=num_classes, input_dim=input_dim).to(device)

train_labels = [dataset._label_from_path(p) for p in train_loader.dataset.file_paths]
class_counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)
# Las clases del label_map que todavia no tienen clips quedan con peso 0: si recibieran el peso
# inflado de dividir por 1, el label smoothing gastaria probabilidad en clases que no existen.
presentes = class_counts > 0
class_weights = np.where(presentes, class_counts.sum() / (presentes.sum() * np.where(presentes, class_counts, 1.0)), 0.0)
print(f'Clases con clips: {int(presentes.sum())} de {num_classes} (las demas llevan peso 0)')
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=class_weights_t)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = build_warmup_cosine_scheduler(optimizer, EPOCHS, WARMUP_EPOCHS)
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=0.001)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('\nIniciando entrenamiento (Conformer)...')
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    lr_actual = optimizer.param_groups[0]['lr']
    print(f'Epoca [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {lr_actual:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': dataset.class_names,
            'max_frames': MAX_FRAMES,
            'input_dim': input_dim,
            'landmarks': 'cara_completa_68_normalizada_por_ojos',
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, VIDEO_MODEL_PATH)
        print(f'  -> Nuevo mejor modelo guardado (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%)')

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f'  -> Early stopping en la epoca {epoch+1}.')
        break

print(f'\nListo. Mejor Val Loss: {best_val_loss:.4f} | Modelo en: {VIDEO_MODEL_PATH}')

## 6. Resultados y comparacion con el modelo de solo labios

In [ ]:
from sklearn.metrics import confusion_matrix

epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
ax1.set_title('Convergencia de Perdida (cara completa)')
ax1.set_xlabel('Epocas'); ax1.set_ylabel('Perdida'); ax1.legend(); ax1.grid(True)
ax2.plot(epochs_range, history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'g-', label='Validation Accuracy', linewidth=2)
ax2.set_title('Precision del modelo de cara completa')
ax2.set_xlabel('Epocas'); ax2.set_ylabel('Precision (%)'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
curvas_path = os.path.join(OUTPUTS_DIR, 'cara_completa_curvas.png')
plt.savefig(curvas_path)
plt.show()
print(f'-> Grafico guardado: {curvas_path}')

best_checkpoint = torch.load(VIDEO_MODEL_PATH, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for sequences, labels, lengths in val_loader:
        sequences = sequences.to(device)
        lengths_dev = lengths.to(device)
        mask = make_padding_mask(lengths_dev, sequences.shape[1], device)
        outputs = model(sequences, src_key_padding_mask=mask)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

total = len(all_labels)
top1_acc = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / total
print(f'Top-1 accuracy (val): {top1_acc:.2f}%')

cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(dataset.class_names))))
plt.figure(figsize=(max(8, len(dataset.class_names) * 0.35), max(6, len(dataset.class_names) * 0.35)))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(dataset.class_names)), dataset.class_names, rotation=90, fontsize=8)
plt.yticks(range(len(dataset.class_names)), dataset.class_names, fontsize=8)
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.title(f'Matriz de confusion (cara completa, val) -- top1 {top1_acc:.1f}%')
plt.colorbar()
plt.tight_layout()
cm_path = os.path.join(OUTPUTS_DIR, 'cara_completa_matriz_confusion.png')
plt.savefig(cm_path)
plt.show()
print(f'-> Matriz de confusion guardada: {cm_path}')

In [ ]:
all_logits = []
model.eval()
with torch.no_grad():
    for sequences, labels, lengths in val_loader:
        sequences = sequences.to(device)
        mask = make_padding_mask(lengths.to(device), sequences.shape[1], device)
        all_logits.append(model(sequences, src_key_padding_mask=mask).cpu())
logits = torch.cat(all_logits)
etiquetas = torch.tensor(all_labels)
top1 = 100.0 * (logits.argmax(1) == etiquetas).float().mean().item()
top3 = 100.0 * (logits.topk(3, dim=1).indices == etiquetas.unsqueeze(1)).any(1).float().mean().item()
es_correcto = lambda idx: dataset.class_names[idx].endswith('_correcto')
pred = logits.argmax(1).tolist()
binaria = 100.0 * sum(es_correcto(p) == es_correcto(l) for p, l in zip(pred, all_labels)) / len(all_labels)

REFERENCIA_LABIOS = {'top1': 67.17, 'top3': 92.46, 'binaria': 75.98}
n_clases = len(dataset.class_names)
if n_clases == 58:
    print(f"{'':28s} {'Solo labios':>12s} {'Cara completa':>14s} {'Diferencia':>11s}")
    for nombre, clave, valor in (('Top-1 (clase exacta, 58)', 'top1', top1), ('Top-3', 'top3', top3), ('Bien o mal (binaria)', 'binaria', binaria)):
        ref = REFERENCIA_LABIOS[clave]
        print(f'{nombre:28s} {ref:11.2f}% {valor:13.2f}% {valor - ref:+10.2f}')
else:
    print(f'Cara completa ({n_clases} clases): Top-1 {top1:.2f}% | Top-3 {top3:.2f}% | Bien o mal {binaria:.2f}%')
    print('Las cifras del modelo de solo labios (67.17 % / 92.46 % / 75.98 %) son de 58 clases con el dataset completo: con otras clases u otra cantidad de clips NO son comparables.')
    print('Para comparar justo: entrena el modelo de labios con Entrenar_Modelos_FonoKids.ipynb sobre los MISMOS clips (sus landmarks de boca) y compara con estas cifras.')
print(f'\nClips de validacion: {len(all_labels)} | Entrada: {input_dim} numeros por frame')

with open(os.path.join(OUTPUTS_DIR, 'cara_completa_resultados.json'), 'w', encoding='utf-8') as f:
    json.dump({'top1': top1, 'top3': top3, 'binaria': binaria, 'referencia_labios': REFERENCIA_LABIOS,
               'clips_validacion': len(all_labels), 'input_dim': int(input_dim), 'clases': len(dataset.class_names)}, f, indent=2)
print('Resultados guardados en', os.path.join(OUTPUTS_DIR, 'cara_completa_resultados.json'))

## Listo

Quedo guardado:

- `modelos_cara_completa/best_cara_completa.pth` -- modelo de video con toda la cara (junto a tu carpeta de videos).
- `resultados_cara_completa/` -- curvas, matriz de confusion y `cara_completa_resultados.json`.

**Como usar el resultado:** si mejora al modelo de solo labios, el servidor (`server/main.py`) todavia habria que adaptarlo para extraer los 68 puntos con la misma normalizacion antes de cargar este modelo; mientras tanto `models/best.pth` sigue funcionando igual. Si no mejora, queda como resultado documentado para la tesis.